# NB1: Ground Truth Comparison

**Narrative Framing**
* What question this notebook addresses: This notebook implements the specified sections to address the statistical analysis of the ground truth transition and performance impact.
* Which GT is being used as primary: Platinum GT
* Which KL1 strategy is active: Explicitly switchable (Default Strategy A - Exclusion)
* Central Narrative: Public medical imaging repository labels contain systematic, directionally biased noise that is invisible without platinum standard validation, and this noise quantifiably corrupts the performance metrics of annotation studies, AI benchmarks, and psychometric predictors derived from them.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, f1_score, brier_score_loss
from statsmodels.stats.contingency_tables import mcnemar
import pingouin as pg
import warnings
import helpers

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Load Data
KL1_STRATEGY = 'clinical' 
df_full = helpers.load_data(KL1_STRATEGY)
df_part = helpers.participant_summary(df_full)
df_img = helpers.image_summary(df_full)

print("Setup Complete: Data loaded using 'clinical' strategy to preserve KL1 comparisons.")

mkdir -p failed for path /Users/martonbaltay/.matplotlib: [Errno 1] Operation not permitted: '/Users/martonbaltay/.matplotlib'


Matplotlib created a temporary cache directory at /var/folders/1g/t32yf_mn5pd5ymkl2yywtctr0000gn/T/matplotlib-gmjnnkea because there was an issue with the default path (/Users/martonbaltay/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Matplotlib is building the font cache; this may take a moment.


Setup Complete: Data loaded using 'clinical' strategy to preserve KL1 comparisons.


## Section 1 — AI model performance under both GTs

In [2]:
img_df = df_full.drop_duplicates('trial_image_name')

o_acc = accuracy_score(img_df['gt_original_binary'], img_df['ai_prediction'])
p_acc = accuracy_score(img_df['gt_plat_binary'], img_df['ai_prediction'])

print("AI Accuracy under Original GT:", o_acc)
print("AI Accuracy under Platinum GT:", p_acc)

tbl = pd.crosstab(img_df['gt_original_binary'] == img_df['ai_prediction'], 
                  img_df['gt_plat_binary'] == img_df['ai_prediction'])
print("\nMcNemar Test for AI performance shift:")
print(mcnemar(tbl, exact=False, correction=True))

AI Accuracy under Original GT: 0.7
AI Accuracy under Platinum GT: 0.7

McNemar Test for AI performance shift:
pvalue      0.7892680261342813
statistic   0.07142857142857142


## Section 2 — Human performance under both GTs

In [3]:
h_o_acc = accuracy_score(df_full['gt_original_binary'], df_full['final_decision'])
h_p_acc = accuracy_score(df_full['gt_plat_binary'], df_full['final_decision'])

print("Human Accuracy under Original GT:", h_o_acc)
print("Human Accuracy under Platinum GT:", h_p_acc)

tbl_h = pd.crosstab(df_full['gt_original_binary'] == df_full['final_decision'], 
                    df_full['gt_plat_binary'] == df_full['final_decision'])
print("\nMcNemar Test for Human performance shift:")
print(mcnemar(tbl_h, exact=False, correction=True))

inversions = df_full[(df_full['gt_original_binary'] == df_full['final_decision']) & 
                     (df_full['gt_plat_binary'] != df_full['final_decision'])]
print(f"\nInversions (Cases correctly labeled under Original GT but INCORRECT under Platinum): {len(inversions)}")

Human Accuracy under Original GT: 0.6290756302521009
Human Accuracy under Platinum GT: 0.6838655462184874

McNemar Test for Human performance shift:
pvalue      1.6868717144318078e-15
statistic   63.40036014405762

Inversions (Cases correctly labeled under Original GT but INCORRECT under Platinum): 670


## Section 3 — Reliance metric recomputation

In [4]:
ai_cond = df_full[df_full['condition'] == 'ai'].copy()

ai_cond['ai_correct_orig'] = ai_cond['ai_prediction'] == ai_cond['gt_original_binary']
rel_o = pd.crosstab(ai_cond['ai_correct_orig'], ai_cond['final_decision'] == ai_cond['ai_prediction'])

rel_p = pd.crosstab(ai_cond['ai_correct_plat'], ai_cond['final_decision'] == ai_cond['ai_prediction'])

print("Reliance Matrix (Original GT):")
print("Index: AI Correct, Column: Human Followed AI")
print(rel_o)
print("\nReliance Matrix (Platinum GT):")
print(rel_p)

Reliance Matrix (Original GT):
Index: AI Correct, Column: Human Followed AI
col_0            False  True 
ai_correct_orig              
False              443    472
True               593   1542

Reliance Matrix (Platinum GT):
col_0            False  True 
ai_correct_plat              
False              493    422
True               543   1592


## Section 4 — The accuracy paradox demonstration

In [5]:
fig = go.Figure(data=[
    go.Bar(name='Original GT', x=['AI', 'Human'], y=[o_acc, h_o_acc]),
    go.Bar(name='Platinum GT', x=['AI', 'Human'], y=[p_acc, h_p_acc])
])
fig.update_layout(barmode='group', title='The Accuracy Paradox: Bias in Performance Evaluation')
fig.show()